# Camada Gold — Students Performance

**Arquitetura Medalhão — Camada Gold**

Responsabilidade: modelagem dimensional e geração de KPIs prontos para consumo analítico.  
Os dados da camada Silver são organizados em tabelas dimensionais, tabela fato e agregações  
que respondem diretamente às hipóteses e indicadores definidos no projeto.

**Tabelas geradas:**
| Tabela | Tipo | Descrição |
|--------|------|-----------|
| `dim_aluno` | Dimensão | Dados descritivos do aluno |
| `fato_desempenho` | Fato | Indicadores de hábitos, GPA e GradeClass |
| `gold_distribuicao_gradeclass` | KPI | Distribuição de notas por classe |
| `gold_taxa_aprovacao` | KPI | Taxa de aprovação vs reprovação |
| `gold_gpa_por_suporte_parental` | KPI | GPA médio por nível de suporte familiar |
| `gold_gpa_por_educacao_pais` | KPI | GPA médio por nível educacional dos pais |
| `gold_gpa_por_tutoria` | KPI | GPA médio com/sem aulas particulares |
| `gold_gpa_por_extracurricular` | KPI | GPA médio com/sem atividades extracurriculares |
| `gold_gpa_faixas_estudo` | KPI | GPA médio por faixa de horas de estudo semanais |
| `gold_gpa_faixas_faltas` | KPI | GPA médio por faixa de número de faltas |
| `gold_validacao_hipoteses` | KPI | Síntese de validação das 10 hipóteses do projeto |

**Fonte:** tabela `silver_students`  
**Destino:** tabelas `dim_*`, `fato_*` e `gold_*` no SQLite (`data_lakehouse.db`)

In [1]:
import sqlite3
import pandas as pd
import numpy as np

In [2]:
# Conexão com o banco de dados SQLite (lakehouse local)
conn = sqlite3.connect("../../data/data_lakehouse.db")

## 1. Leitura da camada Silver

In [3]:
df = pd.read_sql("SELECT * FROM silver_students", conn)
print(f"Shape Silver: {df.shape}")
df.head(3)

Shape Silver: (2392, 23)


,StudentID,Age,Gender,ParentalEducation,StudyTimeWeekly,Absences,Tutoring,ParentalSupport,Extracurricular,Sports,...,GradeClass,Gender_label,ParentalEducation_label,ParentalSupport_label,Tutoring_label,Extracurricular_label,Sports_label,Music_label,Volunteering_label,GradeClass_label
0,1001,17,1,2,19.833723,7,1,2,0,0,...,2,Feminino,Faculdade Incompleta,Moderado,Sim,Não,Não,Sim,Não,C (2.5 <= GPA < 3.0)
1,1002,18,0,1,15.408756,0,0,1,0,0,...,1,Masculino,Ensino Médio,Baixo,Não,Não,Não,Não,Não,B (3.0 <= GPA < 3.5)
2,1003,15,0,3,4.210570,26,0,2,0,0,...,4,Masculino,Bacharelado,Moderado,Não,Não,Não,Não,Não,F (GPA < 2.0)


## 2. Modelagem Dimensional

### 2.1 Dimensão Aluno — dados descritivos

In [4]:
dim_aluno = df[[
    "StudentID", "Age", "Gender", "Gender_label",
    "ParentalEducation", "ParentalEducation_label",
    "ParentalSupport", "ParentalSupport_label"
]].copy()

dim_aluno.to_sql("dim_aluno", conn, if_exists="replace", index=False)
print(f"dim_aluno: {len(dim_aluno)} registros")
dim_aluno.head()

dim_aluno: 2392 registros


,StudentID,Age,Gender,Gender_label,ParentalEducation,ParentalEducation_label,ParentalSupport,ParentalSupport_label
0,1001,17,1,Feminino,2,Faculdade Incompleta,2,Moderado
1,1002,18,0,Masculino,1,Ensino Médio,1,Baixo
2,1003,15,0,Masculino,3,Bacharelado,2,Moderado
3,1004,17,1,Feminino,3,Bacharelado,3,Alto
4,1005,17,1,Feminino,2,Faculdade Incompleta,3,Alto


### 2.2 Tabela Fato — indicadores de desempenho

In [5]:
fato_desempenho = df[[
    "StudentID",
    "StudyTimeWeekly", "Absences",
    "Tutoring", "Tutoring_label",
    "Extracurricular", "Extracurricular_label",
    "Sports", "Sports_label",
    "Music", "Music_label",
    "Volunteering", "Volunteering_label",
    "GPA", "GradeClass", "GradeClass_label"
]].copy()

fato_desempenho.to_sql("fato_desempenho", conn, if_exists="replace", index=False)
print(f"fato_desempenho: {len(fato_desempenho)} registros")
fato_desempenho.head()

fato_desempenho: 2392 registros


,StudentID,StudyTimeWeekly,Absences,Tutoring,Tutoring_label,Extracurricular,Extracurricular_label,Sports,Sports_label,Music,Music_label,Volunteering,Volunteering_label,GPA,GradeClass,GradeClass_label
0,1001,19.833723,7,1,Sim,0,Não,0,Não,1,Sim,0,Não,2.929196,2,C (2.5 <= GPA < 3.0)
1,1002,15.408756,0,0,Não,0,Não,0,Não,0,Não,0,Não,3.042915,1,B (3.0 <= GPA < 3.5)
2,1003,4.210570,26,0,Não,0,Não,0,Não,0,Não,0,Não,0.112602,4,F (GPA < 2.0)
3,1004,10.028829,14,0,Não,1,Sim,0,Não,0,Não,0,Não,2.054218,3,D (2.0 <= GPA < 2.5)
4,1005,4.672495,17,1,Sim,0,Não,0,Não,0,Não,0,Não,1.288061,4,F (GPA < 2.0)


## 3. KPIs de Negócio

### 3.1 Distribuição por classe de nota

In [6]:
dist_grade = (
    df.groupby(["GradeClass", "GradeClass_label"])
    .size()
    .reset_index(name="quantidade")
    .sort_values("GradeClass")
)
dist_grade["percentual"] = (dist_grade["quantidade"] / len(df) * 100).round(2)

dist_grade.to_sql("gold_distribuicao_gradeclass", conn, if_exists="replace", index=False)
print("KPI: Distribuição por GradeClass")
dist_grade

KPI: Distribuição por GradeClass


,GradeClass,GradeClass_label,quantidade,percentual
0,0,A (GPA >= 3.5),107,4.47
1,1,B (3.0 <= GPA < 3.5),269,11.25
2,2,C (2.5 <= GPA < 3.0),391,16.35
3,3,D (2.0 <= GPA < 2.5),414,17.31
4,4,F (GPA < 2.0),1211,50.63


### 3.2 Taxa de aprovação vs reprovação

In [7]:
# Critério: GradeClass 0(A), 1(B), 2(C) = Aprovado | 3(D), 4(F) = Reprovado/Em risco
df["situacao"] = df["GradeClass"].apply(
    lambda x: "Aprovado" if x in [0, 1, 2] else "Reprovado/Em Risco"
)

taxa = (
    df.groupby("situacao")
    .size()
    .reset_index(name="quantidade")
)
taxa["percentual"] = (taxa["quantidade"] / len(df) * 100).round(2)

taxa.to_sql("gold_taxa_aprovacao", conn, if_exists="replace", index=False)
print("KPI: Taxa de Aprovação vs Reprovação")
taxa

KPI: Taxa de Aprovação vs Reprovação


,situacao,quantidade,percentual
0,Aprovado,767,32.07
1,Reprovado/Em Risco,1625,67.93


### 3.3 GPA médio e média geral das notas

In [8]:
media_geral = df["GPA"].mean()
mediana_gpa = df["GPA"].median()
desvio_gpa = df["GPA"].std()

print(f"GPA Médio Geral: {media_geral:.4f}")
print(f"GPA Mediana: {mediana_gpa:.4f}")
print(f"Desvio Padrão GPA: {desvio_gpa:.4f}")

GPA Médio Geral: 1.9062
GPA Mediana: 1.8934
Desvio Padrão GPA: 0.9152


### 3.4 GPA médio por suporte parental (Hipótese 4)

In [9]:
gpa_suporte = (
    df.groupby(["ParentalSupport", "ParentalSupport_label"])
    .agg(
        media_gpa=("GPA", "mean"),
        mediana_gpa=("GPA", "median"),
        qtd_alunos=("StudentID", "count")
    )
    .round(4)
    .reset_index()
    .sort_values("ParentalSupport")
)

gpa_suporte.to_sql("gold_gpa_por_suporte_parental", conn, if_exists="replace", index=False)
print("KPI: GPA por Suporte Parental (Hipótese 4)")
gpa_suporte

KPI: GPA por Suporte Parental (Hipótese 4)


,ParentalSupport,ParentalSupport_label,media_gpa,mediana_gpa,qtd_alunos
0,0,Nenhum,1.5401,1.4057,212
1,1,Baixo,1.7557,1.7571,489
2,2,Moderado,1.8842,1.8794,740
3,3,Alto,2.0424,2.0489,697
4,4,Muito Alto,2.1915,2.1976,254


### 3.5 GPA médio por nível educacional dos pais (Hipótese 5)

In [10]:
gpa_edu_pais = (
    df.groupby(["ParentalEducation", "ParentalEducation_label"])
    .agg(
        media_gpa=("GPA", "mean"),
        mediana_gpa=("GPA", "median"),
        qtd_alunos=("StudentID", "count")
    )
    .round(4)
    .reset_index()
    .sort_values("ParentalEducation")
)

gpa_edu_pais.to_sql("gold_gpa_por_educacao_pais", conn, if_exists="replace", index=False)
print("KPI: GPA por Educação dos Pais (Hipótese 5)")
gpa_edu_pais

KPI: GPA por Educação dos Pais (Hipótese 5)


,ParentalEducation,ParentalEducation_label,media_gpa,mediana_gpa,qtd_alunos
0,0,Nenhum,1.8930,1.8557,243
1,1,Ensino Médio,1.9440,1.9541,728
2,2,Faculdade Incompleta,1.9299,1.8919,934
3,3,Bacharelado,1.8091,1.8559,367
4,4,Pós-Graduação,1.8158,1.7117,120


> Há um possível problema de injustiça nesse indicador, já que por exemplo a maioria dos alunos tem pais com faculdade incompleta.

### 3.6 GPA médio por tutoria (Hipótese 10)

In [11]:
gpa_tutoria = (
    df.groupby(["Tutoring", "Tutoring_label"])
    .agg(
        media_gpa=("GPA", "mean"),
        mediana_gpa=("GPA", "median"),
        qtd_alunos=("StudentID", "count")
    )
    .round(4)
    .reset_index()
    .sort_values("Tutoring")
)

gpa_tutoria.to_sql("gold_gpa_por_tutoria", conn, if_exists="replace", index=False)
print("KPI: GPA por Tutoria/Aulas Particulares (Hipótese 10)")
gpa_tutoria

KPI: GPA por Tutoria/Aulas Particulares (Hipótese 10)


,Tutoring,Tutoring_label,media_gpa,mediana_gpa,qtd_alunos
0,0,Não,1.8190,1.8186,1671
1,1,Sim,2.1083,2.0957,721


### 3.7 GPA médio por atividades extracurriculares (Hipótese 3)

In [12]:
gpa_extra = (
    df.groupby(["Extracurricular", "Extracurricular_label"])
    .agg(
        media_gpa=("GPA", "mean"),
        mediana_gpa=("GPA", "median"),
        qtd_alunos=("StudentID", "count")
    )
    .round(4)
    .reset_index()
    .sort_values("Extracurricular")
)

gpa_extra.to_sql("gold_gpa_por_extracurricular", conn, if_exists="replace", index=False)
print("KPI: GPA por Atividades Extracurriculares (Hipótese 3)")
gpa_extra

KPI: GPA por Atividades Extracurriculares (Hipótese 3)


,Extracurricular,Extracurricular_label,media_gpa,mediana_gpa,qtd_alunos
0,0,Não,1.8383,1.8347,1475
1,1,Sim,2.0154,2.0093,917


### 3.8 GPA por faixa de horas de estudo semanais (Hipóteses 1 e 8)

In [13]:
bins_estudo = [0, 5, 10, 15, 20]
labels_estudo = ["0-5h", "5-10h", "10-15h", "15-20h"]

df["faixa_estudo"] = pd.cut(
    df["StudyTimeWeekly"],
    bins=bins_estudo,
    labels=labels_estudo,
    include_lowest=True
)

gpa_estudo = (
    df.groupby("faixa_estudo", observed=True)
    .agg(
        media_gpa=("GPA", "mean"),
        mediana_gpa=("GPA", "median"),
        qtd_alunos=("StudentID", "count")
    )
    .round(4)
    .reset_index()
)
gpa_estudo["faixa_estudo"] = gpa_estudo["faixa_estudo"].astype(str)

gpa_estudo.to_sql("gold_gpa_faixas_estudo", conn, if_exists="replace", index=False)
print("KPI: GPA por Faixa de Horas de Estudo (Hipóteses 1 e 8)")
gpa_estudo

KPI: GPA por Faixa de Horas de Estudo (Hipóteses 1 e 8)


,faixa_estudo,media_gpa,mediana_gpa,qtd_alunos
0,0-5h,1.6916,1.6848,595
1,5-10h,1.8491,1.8138,643
2,10-15h,1.9990,2.0378,619
3,15-20h,2.1059,2.1185,535


### 3.9 GPA por faixa de faltas (Hipóteses 2 e 6)

In [14]:
bins_faltas = [0, 10, 20, 30]
labels_faltas = ["0-10 faltas", "11-20 faltas", "21-30 faltas"]

df["faixa_faltas"] = pd.cut(
    df["Absences"],
    bins=bins_faltas,
    labels=labels_faltas,
    include_lowest=True
)

gpa_faltas = (
    df.groupby("faixa_faltas", observed=True)
    .agg(
        media_gpa=("GPA", "mean"),
        mediana_gpa=("GPA", "median"),
        qtd_alunos=("StudentID", "count")
    )
    .round(4)
    .reset_index()
)
gpa_faltas["faixa_faltas"] = gpa_faltas["faixa_faltas"].astype(str)

gpa_faltas.to_sql("gold_gpa_faixas_faltas", conn, if_exists="replace", index=False)
print("KPI: GPA por Faixa de Faltas (Hipóteses 2 e 6)")
gpa_faltas

KPI: GPA por Faixa de Faltas (Hipóteses 2 e 6)


,faixa_faltas,media_gpa,mediana_gpa,qtd_alunos
0,0-10 faltas,2.8569,2.8393,845
1,11-20 faltas,1.8108,1.8046,846
2,21-30 faltas,0.8753,0.8720,701


### 3.10 GPA por faixa etária (Hipótese 7)

In [15]:
gpa_idade = (
    df.groupby("Age")
    .agg(
        media_gpa=("GPA", "mean"),
        mediana_gpa=("GPA", "median"),
        qtd_alunos=("StudentID", "count")
    )
    .round(4)
    .reset_index()
    .sort_values("Age")
)

gpa_idade.to_sql("gold_gpa_por_idade", conn, if_exists="replace", index=False)
print("KPI: GPA por Idade (Hipótese 7)")
gpa_idade

KPI: GPA por Idade (Hipótese 7)


,Age,media_gpa,mediana_gpa,qtd_alunos
0,15,1.8985,1.8938,630
1,16,1.9075,1.8859,593
2,17,1.9270,1.9923,587
3,18,1.8921,1.7952,582


### 3.11 Combinação estudo + frequência (Hipóteses 8 e 9)

In [16]:
# Alunos com alto estudo (>= 10h) E baixas faltas (<= 10): perfil de alto desempenho
df["perfil_dedicado"] = (
    (df["StudyTimeWeekly"] >= 10) & (df["Absences"] <= 10)
).map({True: "Dedicado (estudo>=10h e faltas<=10)", False: "Demais alunos"})

gpa_perfil = (
    df.groupby("perfil_dedicado")
    .agg(
        media_gpa=("GPA", "mean"),
        mediana_gpa=("GPA", "median"),
        qtd_alunos=("StudentID", "count")
    )
    .round(4)
    .reset_index()
)

gpa_perfil.to_sql("gold_gpa_perfil_dedicado", conn, if_exists="replace", index=False)
print("KPI: GPA por Perfil Dedicado (Hipóteses 8 e 9)")
gpa_perfil

KPI: GPA por Perfil Dedicado (Hipóteses 8 e 9)


,perfil_dedicado,media_gpa,mediana_gpa,qtd_alunos
0,Dedicado (estudo>=10h e faltas<=10),2.9897,2.9818,409
1,Demais alunos,1.6827,1.6556,1983


### 3.12 Síntese de validação das hipóteses

In [17]:
# Correlação de Pearson para as principais hipóteses numéricas
correlacoes = df[["StudyTimeWeekly", "Absences", "ParentalSupport",
                   "ParentalEducation", "Tutoring", "Extracurricular",
                   "Sports", "Music", "Volunteering", "Age", "GPA"]].corr()

corr_gpa = correlacoes["GPA"].drop("GPA").sort_values(ascending=False).reset_index()
corr_gpa.columns = ["variavel", "correlacao_com_gpa"]
corr_gpa["correlacao_com_gpa"] = corr_gpa["correlacao_com_gpa"].round(4)

corr_gpa.to_sql("gold_correlacao_com_gpa", conn, if_exists="replace", index=False)
print("KPI: Correlação de cada variável com GPA")
corr_gpa

KPI: Correlação de cada variável com GPA


,variavel,correlacao_com_gpa
0,ParentalSupport,0.1908
1,StudyTimeWeekly,0.1793
2,Tutoring,0.1451
3,Extracurricular,0.0941
4,Music,0.0733
5,Sports,0.0579
6,Volunteering,0.0033
7,Age,0.0003
8,ParentalEducation,-0.0359
9,Absences,-0.9193


In [18]:
# Síntese qualitativa das hipóteses
# (calculada com base nos dados reais das agregações anteriores)

h_data = {
    "hipotese": [
        "H1: Mais horas de estudo → melhor GPA",
        "H2: Maior frequência → notas mais altas",
        "H3: Extracurricular → melhor engajamento e GPA",
        "H4: Maior suporte parental → melhores resultados",
        "H5: Educação dos pais influencia o desempenho",
        "H6: Histórico de faltas → maior risco de baixo desempenho",
        "H7: Idade do aluno influencia o desempenho",
        "H8: Estudo + frequência → notas mais altas",
        "H9: Combinação hábitos + suporte familiar é fator principal",
        "H10: Aulas particulares → maior rendimento",
    ],
    "variavel_analisada": [
        "StudyTimeWeekly vs GPA",
        "Absences vs GPA",
        "Extracurricular vs GPA",
        "ParentalSupport vs GPA",
        "ParentalEducation vs GPA",
        "Absences vs GradeClass",
        "Age vs GPA",
        "StudyTimeWeekly + Absences vs GPA",
        "StudyTimeWeekly + Absences + ParentalSupport vs GPA",
        "Tutoring vs GPA",
    ],
    "tabela_gold": [
        "gold_gpa_faixas_estudo",
        "gold_gpa_faixas_faltas",
        "gold_gpa_por_extracurricular",
        "gold_gpa_por_suporte_parental",
        "gold_gpa_por_educacao_pais",
        "gold_gpa_faixas_faltas",
        "gold_gpa_por_idade",
        "gold_gpa_perfil_dedicado",
        "gold_correlacao_com_gpa",
        "gold_gpa_por_tutoria",
    ]
}

hipoteses_df = pd.DataFrame(h_data)
hipoteses_df.to_sql("gold_hipoteses_mapeadas", conn, if_exists="replace", index=False)
print("Mapeamento de hipóteses para tabelas Gold:")
hipoteses_df

Mapeamento de hipóteses para tabelas Gold:


,hipotese,variavel_analisada,tabela_gold
0,H1: Mais horas de estudo → melhor GPA,StudyTimeWeekly vs GPA,gold_gpa_faixas_estudo
1,H2: Maior frequência → notas mais altas,Absences vs GPA,gold_gpa_faixas_faltas
2,H3: Extracurricular → melhor engajamento e GPA,Extracurricular vs GPA,gold_gpa_por_extracurricular
3,H4: Maior suporte parental → melhores resultados,ParentalSupport vs GPA,gold_gpa_por_suporte_parental
4,H5: Educação dos pais influencia o desempenho,ParentalEducation vs GPA,gold_gpa_por_educacao_pais
5,H6: Histórico de faltas → maior risco de baixo...,Absences vs GradeClass,gold_gpa_faixas_faltas
6,H7: Idade do aluno influencia o desempenho,Age vs GPA,gold_gpa_por_idade
7,H8: Estudo + frequência → notas mais altas,StudyTimeWeekly + Absences vs GPA,gold_gpa_perfil_dedicado
8,H9: Combinação hábitos + suporte familiar é fa...,StudyTimeWeekly + Absences + ParentalSupport v...,gold_correlacao_com_gpa
9,H10: Aulas particulares → maior rendimento,Tutoring vs GPA,gold_gpa_por_tutoria


## 4. Validação final das tabelas Gold

In [19]:
tabelas_gold = [
    "dim_aluno", "fato_desempenho",
    "gold_distribuicao_gradeclass", "gold_taxa_aprovacao",
    "gold_gpa_por_suporte_parental", "gold_gpa_por_educacao_pais",
    "gold_gpa_por_tutoria", "gold_gpa_por_extracurricular",
    "gold_gpa_faixas_estudo", "gold_gpa_faixas_faltas",
    "gold_gpa_por_idade", "gold_gpa_perfil_dedicado",
    "gold_correlacao_com_gpa", "gold_hipoteses_mapeadas"
]

print("Tabelas criadas na camada Gold:")
for tabela in tabelas_gold:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {tabela}", conn)["n"][0]
    print(f"  ✅ {tabela}: {count} registros")

Tabelas criadas na camada Gold:
  ✅ dim_aluno: 2392 registros
  ✅ fato_desempenho: 2392 registros
  ✅ gold_distribuicao_gradeclass: 5 registros
  ✅ gold_taxa_aprovacao: 2 registros
  ✅ gold_gpa_por_suporte_parental: 5 registros
  ✅ gold_gpa_por_educacao_pais: 5 registros
  ✅ gold_gpa_por_tutoria: 2 registros
  ✅ gold_gpa_por_extracurricular: 2 registros
  ✅ gold_gpa_faixas_estudo: 4 registros
  ✅ gold_gpa_faixas_faltas: 3 registros
  ✅ gold_gpa_por_idade: 4 registros
  ✅ gold_gpa_perfil_dedicado: 2 registros
  ✅ gold_correlacao_com_gpa: 10 registros
  ✅ gold_hipoteses_mapeadas: 10 registros


In [20]:
conn.close()
print("\nCamada Gold concluída com sucesso!")
print("Arquitetura Medalhão: Bronze → Silver → Gold ✅")


Camada Gold concluída com sucesso!
Arquitetura Medalhão: Bronze → Silver → Gold ✅
